In [1]:
#LCMV beamforming for deep source localization
import os.path as op
import matplotlib
import matplotlib.pyplot as plt
import mne
import numpy as np
import joblib
from mne.beamformer import apply_lcmv, make_lcmv
from mne import make_forward_solution, setup_source_space, setup_volume_source_space

In [2]:
path = '/Users/immlab/Desktop/IMM-Lab'
meg_path = op.join(path, 'MEG')
mri_path = op.join(path, 'MRI')

In [19]:
#Create a source space including cerebellum for fsaverage: 

average_sub = subj = 'fsaverage' #Change this to vml_avg_child to setup averaged volume src for children
bem_dir = op.join(mri_path, subj, 'bem')
fname_aseg = op.join(mri_path, subj, 'mri', 'aseg.mgz')

src_to = mne.read_source_spaces(op.join(mri_path, average_sub, 'bem', f'{average_sub}-ico-5-src.fif'))#change to oct6 for vml_avg_child

model = mne.make_bem_model(subject=subj, ico=5, conductivity=(0.3,),
                           subjects_dir=mri_path)

mne.write_bem_surfaces(op.join(mri_path, subj, 'bem', f'{subj}-bem-model.fif'), model, overwrite=True)

fname_model = op.join(bem_dir, f"{subj}-bem-model.fif")

labels_vol = [ 
    'Right-Cerebellum-Cortex',
    'Right-Cerebellum-White-Matter',
    'Left-Cerebellum-Cortex',
    'Left-Cerebellum-White-Matter'
]

fs_avg_vol_src = setup_volume_source_space(
    subj,
    mri = fname_aseg,
    pos = 5.0,
    bem = fname_model, 
    add_interpolator = True,
    volume_label = labels_vol,
    subjects_dir = mri_path
)

src_to += fs_avg_vol_src
src_to.save(op.join(mri_path, subj, 'bem', f'{subj}-mixed-src.fif'), overwrite=True)
src_to.plot(subjects_dir=mri_path)

    Reading a source space...
    [done]
    Reading a source space...
    [done]
    2 source spaces read
Creating the BEM geometry...
Going from 5th to 5th subdivision of an icosahedron (n_tri: 20480 -> 20480)
inner skull CM is  -0.53 -21.10   6.21 mm
Surfaces passed the basic topology checks.
Complete.

Overwriting existing file.
BEM              : /Users/immlab/Desktop/IMM-Lab/MRI/fsaverage/bem/fsaverage-bem-model.fif
grid                  : 5.0 mm
mindist               : 5.0 mm
MRI volume            : /Users/immlab/Desktop/IMM-Lab/MRI/fsaverage/mri/aseg.mgz

Reading /Users/immlab/Desktop/IMM-Lab/MRI/fsaverage/mri/aseg.mgz...

Loaded inner skull from /Users/immlab/Desktop/IMM-Lab/MRI/fsaverage/bem/fsaverage-bem-model.fif (10242 nodes)
Surface CM = (  -0.5  -21.1    6.2) mm
Surface fits inside a sphere with radius   98.3 mm
Surface extent:
    x =  -75.3 ...   76.3 mm
    y = -113.4 ...   75.0 mm
    z =  -72.4 ...   88.2 mm
Grid extent:
    x =  -80.0 ...   80.0 mm
    y = -115.0 .

[Parallel(n_jobs=1)]: Done   0 out of   1 | elapsed:    5.3s remaining:    5.3s


KeyboardInterrupt: 

In [ ]:
conditions = ['right no-rotation', 'left no-rotation', 'right rotation' ,'left rotation']
conds2 = ['run 1 ', 'run 2 ', 'run 3', 'run 4', 'run 5 ']
#subjects = ['vml_meg_011', 'vml_meg_012', 'vml_meg_013', 'vml_meg_014', 'vml_meg_016', 'vml_meg_018', 
            # 'vml_meg_019', 'vml_meg_021', 'vml_meg_022', 'vml_meg_023', 'vml_meg_024', 'vml_meg_025', 
            # 'vml_meg_026', 'vml_meg_030', 'vml_meg_031']
#subs = ['011', '012', '013', '014', '016', '018', '019',  '021', '022', '023', '024', '025', '026', '030', '031'] #VML adults

subjects = ['vml_meg_002', 'vml_meg_003', 'vml_meg_004', 'vml_meg_006', 
            'vml_meg_009', 'vml_meg_010', 'vml_meg_015', 'vml_meg_016', 
            'vml_meg_027', 'vml_meg_028', 'vml_meg_029']

subs = ['002', '003', '004', '006', '009', '010', '015', '016', '027', '028', '029']#Children

for s in subs:
    print(s)
    #here is where I need the correct source space for each subject 
    fwd = mne.read_forward_solution(op.join(meg_path, f'vml_meg_{s}', f'{s}-cerebellar-fw-solution-run.fif'), verbose=False)
    src = fwd['src']

    morph = mne.compute_source_morph( 
            src = src,
            subject_from=f"vml_mri_{s}",
            subject_to = 'fsaverage',
            src_to=src_to, #fsaverage's volumetric src
            subjects_dir=mri_path,
            # niter_sdr=[5, 5,  2],
            # niter_affine=[5, 5, 2],
            verbose=False
        )
    
    morph_mat = morph.compute_vol_morph_mat()
    morph.save(op.join(meg_path, f'vml_meg_{s}', f'{s}-morph'), overwrite=True)
    morph = mne.read_source_morph(op.join(meg_path, f'vml_meg_{s}', f'{s}-morph-morph.h5'))

    for cond in conds2: 
        stc = mne.read_source_estimate(op.join(meg_path, f'vml_meg_{s}', f'DS_{s}_cond-{cond}-stc.h5')) 
        morphed_stc = morph.apply(stc)
        morphed_stc.save(op.join(meg_path, f'vml_meg_{s}', f'morphed_DS_{s}_cond-{cond}-stc.h5'), overwrite=True)

002


/var/folders/wy/8whx0tpj43sgygf2n0g0f5340000gn/T/ipykernel_1172/1459652259.py:17: RuntimeWarning: This filename (/Users/immlab/Desktop/IMM-Lab/MEG/vml_meg_002/002-cerebellar-fw-solution-run.fif) does not conform to MNE naming conventions. All forward files should end with -fwd.fif, -fwd.fif.gz, _fwd.fif, _fwd.fif.gz, -fwd.h5 or _fwd.h5
  fwd = mne.read_forward_solution(op.join(meg_path, f'vml_meg_{s}', f'{s}-cerebellar-fw-solution-run.fif'), verbose=False)
/Users/immlab/Desktop/IMM-Lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing sparse volumetric morph matrix (will take some time...)


100%|██████████| Vertex : 1102/1102 [00:59<00:00,   18.52it/s]


003


...

100%|██████████| Vertex : 1000/1000 [00:53<00:00,   18.81it/s]


029


/var/folders/wy/8whx0tpj43sgygf2n0g0f5340000gn/T/ipykernel_1172/1459652259.py:17: RuntimeWarning: This filename (/Users/immlab/Desktop/IMM-Lab/MEG/vml_meg_029/029-cerebellar-fw-solution-run.fif) does not conform to MNE naming conventions. All forward files should end with -fwd.fif, -fwd.fif.gz, _fwd.fif, _fwd.fif.gz, -fwd.h5 or _fwd.h5
  fwd = mne.read_forward_solution(op.join(meg_path, f'vml_meg_{s}', f'{s}-cerebellar-fw-solution-run.fif'), verbose=False)


Computing sparse volumetric morph matrix (will take some time...)


100%|██████████| Vertex : 963/963 [00:51<00:00,   18.56it/s]


In [ ]:
print(conds2)
print(conditions)
print(conds2)

['run 1 ', 'run 2 ', 'run 3', 'run 4', 'run 5 ']
['right no-rotation', 'left no-rotation', 'right rotation', 'left rotation']
['run 1 ', 'run 2 ', 'run 3', 'run 4', 'run 5 ']


In [ ]:
#Average source estimates across a group of participants (adults)

conds2 = ['run 1', 'run 2', 'run 3', 'run 4', 'run 5 ']

for cond in conds2: 
    stc_p011 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_011', f'morphed_DS_011_cond-{cond}-stc.h5'))
    stc_p012 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_012', f'morphed_DS_012_cond-{cond}-stc.h5'))
    stc_p013 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_013', f'morphed_DS_013_cond-{cond}-stc.h5'))
    stc_p014 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_014', f'morphed_DS_014_cond-{cond}-stc.h5'))
    stc_p018 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_018', f'morphed_DS_018_cond-{cond}-stc.h5'))
    stc_p019 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_019', f'morphed_DS_019_cond-{cond}-stc.h5'))
    stc_p021 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_021', f'morphed_DS_021_cond-{cond}-stc.h5'))
    stc_p022 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_022', f'morphed_DS_022_cond-{cond}-stc.h5'))
    stc_p023 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_023', f'morphed_DS_023_cond-{cond}-stc.h5'))
    stc_p024 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_024', f'morphed_DS_024_cond-{cond}-stc.h5'))
    stc_p025 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_025', f'morphed_DS_025_cond-{cond}-stc.h5'))
    stc_p026 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_026', f'morphed_DS_026_cond-{cond}-stc.h5'))
    stc_p030 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_030', f'morphed_DS_030_cond-{cond}-stc.h5'))
    stc_p031 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_031', f'morphed_DS_031_cond-{cond}-stc.h5'))
    
    all_stcs = [stc_p011, stc_p012, stc_p013, stc_p014, stc_p018, stc_p019, stc_p021, stc_p022, stc_p023, stc_p024, stc_p025, stc_p026, stc_p030, stc_p031] #VML adults

    # Initialize an array to store the data for all participants
    data_array = np.zeros((len(all_stcs), all_stcs[0].data.shape[0], all_stcs[0].data.shape[1]))
    # NP.ZEROS(6, 21683, 1701)

    # Check if all source spaces have the same vertices for both hemispheres or volume
    for i, stc in enumerate(all_stcs):
        if stc.data.shape != data_array[0].shape:
            raise ValueError(f"Shape mismatch for subject {i}: {stc.data.shape} vs {data_array[0].shape}")

    # Fill the data array with source estimate data from each participant
    for i, stc in enumerate(all_stcs):
        data_array[i, :, :] = stc.data

    # Calculate the average across participants
    average_data = np.mean(data_array, axis=0)

    #Check that the shape of average data matches the shape of the stc data:
    print(np.shape(average_data), 'is the same as', np.shape(all_stcs[0].data))

    # Create a new SourceEstimate with the averaged data
    average_stc = mne.MixedSourceEstimate(average_data, vertices=all_stcs[0].vertices, tmin=all_stcs[0].times[0], tstep=np.diff(all_stcs[0].times)[0], subject='fsaverage')

    # Save or plot the averaged source estimate
    average_stc.save(op.join(meg_path, f'VML_adult_morphed-averaged_DS_{cond}'), ftype='h5', overwrite=True)


#Average source estimates across a group of participants (children)

for cond in conds2: 
    # Load STC plots for mixed source space
    stc_p002 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_002', f'morphed_DS_002_cond-{cond}-stc.h5'))
    stc_p003 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_003', f'morphed_DS_003_cond-{cond}-stc.h5'))
    stc_p004 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_004', f'morphed_DS_004_cond-{cond}-stc.h5'))
    stc_p006 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_006', f'morphed_DS_006_cond-{cond}-stc.h5'))
    stc_p009 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_009', f'morphed_DS_009_cond-{cond}-stc.h5'))
    stc_p010 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_010', f'morphed_DS_010_cond-{cond}-stc.h5'))
    stc_p015 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_015', f'morphed_DS_015_cond-{cond}-stc.h5'))
    stc_p016 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_016', f'morphed_DS_016_cond-{cond}-stc.h5'))
    stc_p027 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_027', f'morphed_DS_027_cond-{cond}-stc.h5'))
    stc_p028 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_028', f'morphed_DS_028_cond-{cond}-stc.h5'))
    stc_p029 = mne.read_source_estimate(op.join(meg_path, 'vml_meg_029', f'morphed_DS_029_cond-{cond}-stc.h5'))
        
    all_stcs = [stc_p002, stc_p003, stc_p004, stc_p006, stc_p009, stc_p010, stc_p015, stc_p016, stc_p027, stc_p028, stc_p029] #VML children

    # Initialize an array to store the data for all participants
    data_array = np.zeros((len(all_stcs), all_stcs[0].data.shape[0], all_stcs[0].data.shape[1]))
    # NP.ZEROS(6, 21683, 1701)

    # Check if all source spaces have the same vertices for both hemispheres or volume
    for i, stc in enumerate(all_stcs):
        if stc.data.shape != data_array[0].shape:
            raise ValueError(f"Shape mismatch for subject {i}: {stc.data.shape} vs {data_array[0].shape}")

    # Fill the data array with source estimate data from each participant
    for i, stc in enumerate(all_stcs):
        data_array[i, :, :] = stc.data

    # Calculate the average across participants
    average_data = np.mean(data_array, axis=0)

    #Check that the shape of average data matches the shape of the stc data:
    print(np.shape(average_data), 'is the same as', np.shape(all_stcs[0].data))

    # Create a new SourceEstimate with the averaged data
    average_stc = mne.MixedSourceEstimate(average_data, vertices=all_stcs[0].vertices, tmin=all_stcs[0].times[0], tstep=np.diff(all_stcs[0].times)[0], subject='fsaverage')

    # Save or plot the averaged source estimate
    average_stc.save(op.join(meg_path, f'VML_adult_morphed-averaged_DS_{cond}'), ftype='h5', overwrite=True)


(21683, 1701) is the same as (21683, 1701)
(21683, 1701) is the same as (21683, 1701)
(21683, 1701) is the same as (21683, 1701)
(21683, 1701) is the same as (21683, 1701)
(21683, 1701) is the same as (21683, 1701)


In [ ]:
#Plot the averaged STCs (adults)

titles = ['Baseline', 'Early-Learning', 'Late Learning', 'Washout', 'Re-Learning']
conds = ['run 1', 'run 2', 'run 3', 'run 4', 'run 5']
conditions = ['right no-rotation', 'left no-rotation', 'right rotation', 'left rotation']
i = 0
src_to = mne.read_source_spaces(op.join(mri_path, 'vml_avg_child', 'bem', 'vml_avg_child-mixed-src.fif'))
adult_src = mne.read_source_spaces(op.join(mri_path, 'fsaverage', 'bem', 'fsaverage-mixed-src.fif'), verbose=False)

for cond in conds:
    print(cond)
    avg_stc = mne.read_source_estimate(op.join(meg_path, f'VML_adult_morphed-averaged_DS_{cond}-stc.h5'))
    brain = avg_stc.plot(src=adult_src, subject='fsaverage', subjects_dir=mri_path, surface='white', hemi='split', views=['lat'], smoothing_steps=10)
    brain.add_text(text=f'{titles[i]}-ADULT-AVERAGED-STC', font_size=20, color='white', x=0.2, y=0.9)
    
    # brain.save_movie(
    #     filename=(op.join(meg_path, f'Child_Avg_{titles[i]}.mov')),
    #     time_dilation=60,  # Adjust this value to control the speed of the video
    #     tmin=0,  # Start time
    #     tmax=0.5,  # End time
    #     framerate=24,  # Frames per second
    #     interpolation='linear'
    # )
    # print('saved movie for child', titles[i])
    # # Close the Brain object to release resources
    # brain.close()
    
    i += 1 

#clim=dict(kind="value", lims=[0.02, 0.03, 0.04]),


    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    6 source spaces read
run 1
Using control points [0.04024545 0.04472066 0.07862167]
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
run 2
Using c

In [ ]:
#Plot the averaged STCs (children)

titles = ['Baseline', 'Early-Learning', 'Late Learning', 'Washout', 'Re-Learning']
conds = ['run 1', 'run 2', 'run 3', 'run 4', 'run 5']
conditions = ['right no-rotation', 'left no-rotation', 'right rotation', 'left rotation']
i = 0
j = 0 
src_to = mne.read_source_spaces(op.join(mri_path, 'vml_avg_child', 'bem', 'vml_avg_child-mixed-src.fif'))
adult_src = mne.read_source_spaces(op.join(mri_path, 'fsaverage', 'bem', 'fsaverage-mixed-src.fif'), verbose=False)

for cond in conds:
    print(cond)
    avg_stc = mne.read_source_estimate(op.join(meg_path, f'VML_children_morphed-averaged_DS_{cond}-stc.h5'))
    brain = avg_stc.plot(src=adult_src, subject='fsaverage', subjects_dir=mri_path, surface='white', hemi='split', views=['lat'], smoothing_steps=10)
    brain.add_text(text=f'{titles[i]}-CHILDREN-AVERAGED-STC', font_size=20, color='white', x=0.2, y=0.9)
    
    # brain.save_movie(
    #     filename=(op.join(meg_path, f'Child_Avg_{titles[i]}.mov')),
    #     time_dilation=60,  # Adjust this value to control the speed of the video
    #     tmin=0,  # Start time
    #     tmax=0.5,  # End time
    #     framerate=24,  # Frames per second
    #     interpolation='linear'
    # )
    # print('saved movie for child', titles[i])
    # # Close the Brain object to release resources
    # brain.close()
    
    i += 1 

#clim=dict(kind="value", lims=[0.02, 0.03, 0.04]),


    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    Reading a source space...
    [done]
    6 source spaces read
run 1
Using control points [0.05961207 0.0665456  0.1200397 ]
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
run 2
Using c